# 03. 화학물질 특성 예측 모델 구축

- 앞선 실습에서 HIF-2a 데이터를 모으고(01), 정리·분자 특성값을 계산함(02)
- 이번 시간에는 그 데이터로 **실제 예측 모델을 만들어 봄**
  - **분류 모델**: 화합물이 **활성(Active) / 비활성(Inactive)** 인지 맞힘
  - **회귀 모델**: **활성 점수(Score)를 숫자로** 예측함


---
## 0. 환경설정

- 실습에 필요한 도구(라이브러리)를 준비함
- Colab 환경에서는 대부분 이미 설치되어 있어 설치는 금방 끝남


In [ ]:
# [1] 필요한 라이브러리 설치 (Colab엔 대부분 이미 설치돼 있어서 금방 끝납니다)
# !pip install scikit-learn

# [2] 데이터를 다루는 도구
import pandas as pd            # 표(엑셀 같은 데이터)를 다루는 도구
import numpy as np             # 숫자 계산을 도와주는 도구

# [3] 그림(그래프)을 그리는 도구
import matplotlib.pyplot as plt

# [4] 머신러닝(예측 모델) 도구들 - 필요한 것만 골라서 불러옵니다
from sklearn.model_selection import train_test_split          # 데이터를 학습/평가로 나누기
from sklearn.preprocessing import StandardScaler              # 데이터 정규화
from sklearn.ensemble import RandomForestClassifier           # 분류 모델 (랜덤 포레스트)
from sklearn.ensemble import RandomForestRegressor            # 회귀 모델 (랜덤 포레스트)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, classification_report  # 분류 성능 지표
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # 혼동 행렬
from sklearn.metrics import roc_curve, auc                    # ROC 곡선
from sklearn.metrics import precision_recall_curve, average_precision_score  # PR 곡선
from sklearn.metrics import r2_score, mean_squared_error      # 회귀 성능 지표

print('준비 완료! 모든 도구를 불러왔습니다.')

## 1. 데이터 불러오기 & 특성/라벨 준비

- 지난 시간(02)에 만든 `hif2a_preprocessed.csv` 를 불러옴
- 이 파일에는 각 화합물의 **분자 특성값 6개** / **Fingerprint 2048개**와 **활성 정보**가 들어 있음

| 구분 | 기호 | 의미 |
|---|---|---|
| 특성 | X (features) | 모델에게 보여줄 '입력' → 분자 특성값 6개 & Fingerprint 2048개 |
| 라벨 | y (label) | 모델이 맞혀야 할 '정답' → 활성 여부 (Active=1, Inactive=0) |


In [ ]:
# [1] 지난 시간(02 전처리 실습)에서 저장한 CSV 파일을 불러옵니다
dataset_url = "https://raw.githubusercontent.com/SeyongChoi/20260716_Lecture/main/hif2a_preprocessed.csv"
df = pd.read_csv(dataset_url)

# [2] 데이터가 몇 줄, 몇 열인지 확인 (행 개수, 열 개수)
print('데이터 크기:', df.shape)

# [3] 맨 위 5줄을 미리 살펴보기
df.head()

In [ ]:
# [1] 모델의 입력이 될 분자 특성값 6개의 이름 및 Fingerpirnt 열의 이름을 정리해 둡니다
descriptor_cols = ['MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'NumRotatableBonds']
fingerprint_cols = [f'FP_{i}' for i in range(2048)]  # FP_0 ~ FP_2047

feature_cols = descriptor_cols + fingerprint_cols  # 특성값 6개 + 지문 2048개

# [2] 특성(X): 위에서 정한 descriptor_cols와 fingerprint_cols를 합쳐서 feature_cols를 만들고, 이를 이용해 X를 만듭니다
X = df[feature_cols]

# [3] 라벨(y): 'Active'이면 1, 아니면(Inactive) 0 으로 바꿔 줍니다
#     (컴퓨터는 글자보다 숫자를 더 잘 다루기 때문이에요)
y = []

# Active는 1, Inactive는 0으로 변환합니다
for activity in df['PUBCHEM_ACTIVITY_OUTCOME']:
    if activity == 'Active':
        y.append(1)
    else:
        y.append(0)

# 라벨을 pandas Series로 변환합니다
y = pd.Series(y)

#y = (df['PUBCHEM_ACTIVITY_OUTCOME'] == 'Active').astype(int)

# [4] Active(1)와 Inactive(0)가 각각 몇 개인지 세어 봅니다 (데이터 균형 확인)
print('활성/비활성 개수:')
print(y.value_counts())

## 2. 데이터 분할 (학습용 / 평가용) 및 정규화

### 2.1 데이터 분할(학습용 / 평가용)
- 모델을 만들 때 데이터를 두 덩어리로 나눔

| 구분 | 용도 | 비율 |
|---|---|---|
| 학습용(train) | 모델이 공부하는 데 사용 | 80% |
| 평가용(test) | 처음 보는 데이터로 실력 시험 | 20% |

- 이렇게 나누는 이유: 공부한 문제만 잘 푸는 게 아니라 **처음 보는 문제도 잘 푸는지** 확인하려는 것임

In [ ]:
# [1] 데이터를 학습용(80%)과 평가용(20%)으로 나눕니다
#     - test_size=0.2  : 20%를 평가용으로 사용
#     - random_state=42: 실행할 때마다 같은 결과가 나오도록 고정
#     - stratify=y     : Active/Inactive 비율을 비슷하게 유지 (y 라벨 기준으로 층화 샘플링)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# [2] 학습용 데이터의 Active/Inactive 개수 확인
print('[학습용 데이터]')
print('전체 개수:', len(X_train))
print('Inactive(0):', (y_train == 0).sum())
print('Active(1):', (y_train == 1).sum())

# [3] 평가용 데이터의 Active/Inactive 개수 확인
print('\n[평가용 데이터]')
print('전체 개수:', len(X_test))
print('Inactive(0):', (y_test == 0).sum())
print('Active(1):', (y_test == 1).sum())

### 2.2 데이터 정규화 (Normliazation) -  특성별 스케일 맞추기

- 특성(X)마다 값의 **단위와 범위가 제각각**임

| 특성 | 대략적인 값 범위 |
|---|---|
| MolWt (분자량) | 수백 ~ 수천 |
| MolLogP (지용성) | 약 -3 ~ 8 |
| TPSA (극성 표면적) | 0 ~ 수백 |
| NumHDonors / NumHAcceptors / NumRotatableBonds (개수) | 0 ~ 수십 |
| Fingerprint (FP_0 ~ FP_2047) | 0 또는 1 |

- 이렇게 스케일이 다르면, 값이 큰 특성(예: 분자량)이 **실제 중요도와 상관없이** 계산을 지배할 수 있음
- **정규화(표준화)**: 각 열(column)을 **평균 0, 표준편차 1**로 바꿔 모든 특성을 같은 눈금 위에 올려놓음 → 사이킷런의 `StandardScaler`

In [ ]:
### Descriptor 열별 값 분포 확인
# 2행 3열의 그래프 공간을 만듭니다
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# 각 Descriptor의 값 분포를 하나씩 그립니다
for col, ax in zip(descriptor_cols, axes.flatten()):
    ax.hist(X_train[col], bins=30, edgecolor='black')
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

plt.suptitle('Descriptor Distributions Before Standardization', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
### Descriptor 열별 표준화

# 표준화를 수행할 도구를 만듭니다
scaler = StandardScaler()

# 원본 데이터를 유지하기 위해 복사합니다
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# 학습용 데이터의 평균과 표준편차를 이용해 표준화합니다
X_train_scaled[descriptor_cols] = scaler.fit_transform(
    X_train[descriptor_cols]
)

# 학습용 데이터에서 구한 기준을 평가용 데이터에도 적용합니다
X_test_scaled[descriptor_cols] = scaler.transform(
    X_test[descriptor_cols]
)

In [ ]:
### 표준화 후 Descriptor 열별 값 분포 확인

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, ax in zip(descriptor_cols, axes.flatten()):
    ax.hist(X_train_scaled[col], bins=30, edgecolor='black')
    ax.set_title(col)
    ax.set_xlabel('Standardized Value')
    ax.set_ylabel('Count')

plt.suptitle('Descriptor Distributions After Standardization', fontsize=16)
plt.tight_layout()
plt.show()

## 3. 분류 모델 학습하기

- **랜덤 포레스트(Random Forest)** 모델을 사용함

> 랜덤 포레스트: 여러 개의 **결정 트리(나무)** 를 만들고, 각 나무의 예측을 모아 **다수결로 최종 답**을 정하는 방법임 (여러 명이 투표하는 것과 비슷함)

> 🖼️ **[그림 자리]** — 여러 결정 트리 → 다수결 투표 → 최종 예측 개념도


In [ ]:
# [1] 분류 모델을 만듭니다
#     - n_estimators=300 : 나무(결정트리)를 300그루 만들어서 다수결
#     - random_state=42  : 결과 재현을 위한 고정값
model = RandomForestClassifier(n_estimators=300, random_state=42)

# [2] 학습용 데이터로 모델을 '공부'시킵니다 (fit = 학습)
model.fit(X_train_scaled, y_train)

print('모델 학습 완료!')

## 4. 예측 & 성능 평가

- 학습이 끝난 모델에게 **평가용 데이터**를 보여줌
- 얼마나 잘 맞히는지 점수를 매김

### 📌 먼저, 4가지 기본 개념 (혼동 행렬 기준)

- Active(활성)를 **양성(Positive)**, Inactive(비활성)를 **음성(Negative)** 으로 봄

| 기호 | 이름 | 의미 |
|---|---|---|
| **TP** | True Positive | 실제 Active를 **Active로 맞힘** ✅ |
| **TN** | True Negative | 실제 Inactive를 **Inactive로 맞힘** ✅ |
| **FP** | False Positive | 실제 Inactive를 Active라고 **잘못 예측** ❌ |
| **FN** | False Negative | 실제 Active를 Inactive라고 **놓침** ❌ |

### 📊 성능 지표(metric) 정리

| 지표 | 수식 | 한 줄 의미 | 언제 중요한가 |
|---|---|---|---|
| **정확도 (Accuracy)** | $\dfrac{TP + TN}{TP + TN + FP + FN}$ | 전체 중 **맞힌 비율** | 데이터가 **균형** 있을 때 |
| **정밀도 (Precision)** | $\dfrac{TP}{TP + FP}$ | Active라고 **예측한 것 중** 진짜 비율 | **틀린 양성**을 줄이고 싶을 때 |
| **재현율 (Recall)** | $\dfrac{TP}{TP + FN}$ | **실제 Active 중** 찾아낸 비율 | **놓치면 안 될 때** (예: 신약 후보) |
| **F1 점수 (F1-score)** | $2 \cdot \dfrac{Precision \cdot Recall}{Precision + Recall}$ | 정밀도·재현율의 **조화 평균** | 둘의 **균형**을 볼 때 |
| **ROC-AUC** | $\displaystyle\int_{0}^{1} TPR \, d(FPR)$ | ROC 곡선 **아래 넓이** (0.5=랜덤, 1=완벽) | 전반적인 **구분 능력** |
| **Average Precision (AUPRC)** | $\displaystyle\sum_{n} (R_n - R_{n-1}) \, P_n$ | PR 곡선 **아래 넓이** (평균 정밀도) | **불균형 데이터**의 양성 탐지 |

> 참고
> - $TPR$(재현율) $= \dfrac{TP}{TP+FN}$, &nbsp; $FPR = \dfrac{FP}{FP+TN}$
> - $P_n, R_n$: $n$번째 기준점(threshold)에서의 정밀도(Precision)·재현율(Recall)
> - 모든 지표는 **1(=100%)에 가까울수록 좋음** (RMSE 등 오차 지표만 예외)


In [ ]:
# [1] 평가용 데이터에 대해 예측
#     Inactive는 0, Active는 1로 예측합니다
y_pred = model.predict(X_test_scaled)

# [2] Active(1)일 확률을 계산합니다
#     predict_proba() 결과의 두 번째 열[:, 1]이 Active일 확률입니다
y_proba = model.predict_proba(X_test_scaled)[:, 1]


# [3] 주요 평가 지표를 계산합니다

# 정확도: 전체 데이터 중에서 정답을 맞힌 비율
accuracy = accuracy_score(y_test, y_pred)

# 정밀도: Active라고 예측한 것 중 실제로 Active인 비율
precision = precision_score(y_test, y_pred)

# 재현율: 실제 Active 중에서 모델이 Active라고 찾아낸 비율
recall = recall_score(y_test, y_pred)

# F1 점수: 정밀도와 재현율을 함께 고려한 점수
f1 = f1_score(y_test, y_pred)

# ROC-AUC: 모델이 Active와 Inactive를 전반적으로 구분하는 능력
roc_auc = roc_auc_score(y_test, y_proba)

# Average Precision: 불균형 데이터에서 Active 탐지 성능을 평가하는 지표
average_precision = average_precision_score(y_test, y_proba)


# [4] 계산한 평가 지표를 출력합니다
print('정확도(Accuracy):           {:.3f}'.format(accuracy))
print('정밀도(Precision):          {:.3f}'.format(precision))
print('재현율(Recall):             {:.3f}'.format(recall))
print('F1 점수(F1-score):          {:.3f}'.format(f1))
print('ROC-AUC:                   {:.3f}'.format(roc_auc))
print('Average Precision (AUPRC): {:.3f}'.format(average_precision))


# [5] 더 자세한 성능표를 출력합니다
print('\n=== 상세 성능 보고서 ===')

print(classification_report(
    y_test,
    y_pred,
    target_names=['Inactive', 'Active']
))

## 5. 성능 시각화

- 숫자만 보면 감이 잘 안 오므로, 성능을 **그림**으로 확인함

| 그림 | 보는 것 |
|---|---|
| Confusion Matrix (혼동 행렬) | 맞힌 것과 틀린 것을 표로 정리 |
| ROC 곡선 | 활성/비활성을 얼마나 잘 구분하는지 |
| PR 곡선 | 활성(양성)을 찾아내는 능력 |

> 참고: 그래프 축 글자 깨짐(한글 폰트 문제)을 막기 위해 **축 라벨은 영어**로 적음. 제목은 한글로 둬도 됨.


In [ ]:
# === Confusion Matrix (혼동 행렬) ===
# [1] 실제 정답(y_test)과 모델 예측(y_pred)을 비교한 표를 계산
cm = confusion_matrix(y_test, y_pred)

# [2] 한글 폰트 깨짐 방지를 위해 축 라벨은 영어로 표시합니다
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Inactive', 'Active'])

# [3] 그림 그리기 (숫자가 칸 안에 표시됩니다)
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Confusion Matrix')   # 제목
ax.set_xlabel('Predicted label')   # 가로축: 모델이 예측한 값
ax.set_ylabel('True label')        # 세로축: 실제 정답
plt.tight_layout()
plt.show()

In [ ]:
# === ROC 곡선 ===
# [1] 여러 기준점에서의 FPR(가짜 양성 비율)과 TPR(진짜 양성 비율)을 계산
fpr, tpr, _ = roc_curve(y_test, y_proba)

# [2] 곡선 아래 넓이(AUC) 계산 -> 1에 가까울수록 좋은 모델
roc_auc = auc(fpr, tpr)

# [3] 그림 그리기 (축 라벨은 폰트 깨짐 방지를 위해 영어로)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, color='C0', lw=2, label='ROC (AUC = {:.3f})'.format(roc_auc))
# 아무렇게나 찍는 모델의 기준선(대각선, AUC=0.50)
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random (0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# === PR 곡선 (Precision-Recall) ===
# [1] 여러 기준점에서의 정밀도(Precision)와 재현율(Recall)을 계산
precision, recall, _ = precision_recall_curve(y_test, y_proba)

# [2] 곡선의 평균 정밀도(AP) 계산 -> 1에 가까울수록 좋음
ap = average_precision_score(y_test, y_proba)

# [3] 기준선 = 실제 양성(Active) 비율 (아무 정보 없이 예측했을 때의 수준)
baseline = y_test.mean()

# [4] 그림 그리기 (축 라벨은 폰트 깨짐 방지를 위해 영어로)
plt.figure(figsize=(5, 5))
plt.plot(recall, precision, color='C1', lw=2, label='PR (AP = {:.3f})'.format(ap))
plt.axhline(y=baseline, color='gray', lw=1, linestyle='--',
            label='Baseline ({:.3f})'.format(baseline))
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 6. [실습] 회귀 모델로 활성 점수 예측하기

> **직접 해보기 🖐️**
>
> - 지금까지는 Active/Inactive를 맞히는 **분류**를 함
> - 이번에는 활성 점수(`Score`)라는 **숫자 자체를 예측**하는 **회귀** 모델을 직접 완성함
> - 아래 코드의 `______`(빈칸)을 채워 넣음 (그래프 등 나머지는 완성돼 있음)

💡 **힌트 — 분류 vs 회귀 대응**

| 항목 | 분류 | 회귀 |
|---|---|---|
| 모델 | `RandomForestClassifier` | `RandomForestRegressor` |
| 흐름 | `.fit()` → `.predict()` | 동일 |
| 지표 | `accuracy_score` | `r2_score` (1에 가까울수록 좋음) |


In [ ]:
# [1] 회귀의 정답(라벨)은 숫자 점수인 '%Activity at 18.75 uM_Mean' 입니다
y = df['%Activity at 18.75 uM_Mean']

# 두 그래프에 동일한 bin 경계 적용
common_bins = np.linspace(y.min(), y.max(), 31)

# y가 0 이상인 행만 사용
# --> Activty가 음수이면 이상치로 판단하고 제거
valid_mask = df['%Activity at 18.75 uM_Mean'] >= 0

X_reg = X.loc[valid_mask].copy()
y_reg = df.loc[valid_mask, '%Activity at 18.75 uM_Mean'].copy()

print('제거된 음수 row 수:', (df['%Activity at 18.75 uM_Mean'] < 0).sum())
print('회귀에 사용할 전체 데이터 수:', len(y_reg))

# 분포 확인
y.hist(bins=common_bins)
y_reg.hist(bins=common_bins, alpha=0.6)
plt.xlabel('%Activity at 18.75 uM_Mean')
plt.ylabel('Count')
plt.show()

In [ ]:
# [2] 특성(X)은 분류 때와 똑같이 6개 특성을 사용합니다
#     회귀용으로 데이터를 다시 학습/평가로 나눕니다 (숫자 예측이라 stratify는 사용 안 함)
X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
print('회귀 학습용:', len(X_train), '/ 회귀 평가용:', len(X_test))

### Descriptor 열별 표준화

# 표준화를 수행할 도구를 만듭니다
scaler = StandardScaler()

# 원본 데이터를 유지하기 위해 복사합니다
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# 학습용 데이터의 평균과 표준편차를 이용해 표준화합니다
X_train_scaled[descriptor_cols] = scaler.fit_transform(
    X_train[descriptor_cols]
)

# 학습용 데이터에서 구한 기준을 평가용 데이터에도 적용합니다
X_test_scaled[descriptor_cols] = scaler.transform(
    X_test[descriptor_cols]
)

In [ ]:
# ===== 여기 빈칸(______)을 채워 보세요! =====

# [1] 회귀 모델 만들기 (힌트: RandomForestRegressor, n_estimators=300, random_state=42)
reg = ______

# [2] 학습용 데이터로 모델 학습시키기 (힌트: 분류와 똑같이 .fit)
reg.______(X_train, y_train)

# [3] 평가용 데이터로 점수 예측하기 (힌트: .predict)
y_pred = reg.______(X_test)

# [4] 성능 평가하기
#     R2 점수 (1에 가까울수록 좋음)  -- 힌트: r2_score(실제값, 예측값)
r2 = ______(y_test, y_pred)
#     RMSE (0에 가까울수록 좋음) -- 오차의 크기, 이미 완성해 두었습니다
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print('R2 점수: {:.3f}'.format(r2))
print('RMSE   : {:.3f}'.format(rmse))

In [ ]:
# === Parity plot: 실제값 vs 예측값 (이 코드는 이미 완성되어 있습니다) ===
# 점들이 대각선(y=x)에 가까울수록 예측이 정확한 것입니다
plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='C2')   # 실제(가로) vs 예측(세로)

# 완벽하게 맞혔을 때의 기준선 (y = x)
lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())
plt.plot([lo, hi], [lo, hi], color='gray', lw=1, linestyle='--', label='y = x')

plt.xlabel('Actual Score')       # 축 라벨은 폰트 깨짐 방지를 위해 영어
plt.ylabel('Predicted Score')
plt.title('Parity Plot (R2 = {:.3f})'.format(r2))
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# === Residual plot: 잔차(오차) 확인 (선택 사항, 이미 완성) ===
# 잔차 = 예측값 - 실제값. 0을 나타내는 가로선 근처에 고르게 흩어져 있으면 좋습니다
residuals = y_pred - y_test

plt.figure(figsize=(5, 4))
plt.scatter(y_pred, residuals, alpha=0.5, color='C3')
plt.axhline(y=0, color='gray', lw=1, linestyle='--')
plt.xlabel('Predicted Score')
plt.ylabel('Residual (Pred - Actual)')
plt.title('Residual Plot')
plt.tight_layout()
plt.show()

In [ ]:
# === Residual distribution: 잔차(오차) 확인 (선택 사항, 이미 완성) ===
# 잔차 = 예측값 - 실제값. 0중심 대칭이면 좋음
residuals = y_pred - y_test
mean_residual = residuals.mean()

# 잔차 분포를 히스토그램으로 표시
plt.figure(figsize=(6, 4))
plt.hist(
    residuals,
    bins=30,
    color='coral',
    edgecolor='white'
)

# 잔차가 0인 기준선
plt.axvline(
    x=0,
    color='gray',
    linestyle='--',
    linewidth=1
)

# 잔차 평균 표시
plt.axvline(
    x=mean_residual,
    color='darkred',
    linewidth=2,
    label='Mean = {:.2f}'.format(mean_residual)
)
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Count')
plt.title('Residual Distribution')
plt.legend()
plt.tight_layout()
plt.show()

## 7. (심화) DNN으로 회귀 성능 높이기

- 랜덤 포레스트 대신 **DNN(심층 신경망, Deep Neural Network)** 으로 같은 점수를 예측해 봄
- 위 회귀 실습에서 만든 **표준화된 데이터**(`X_train_scaled`, `X_test_scaled`, `y_train`, `y_test`)를 그대로 사용함
- 여러 층(layer)을 쌓아 **더 복잡한 패턴**을 학습 → 랜덤 포레스트와 성능을 비교함

> DNN: 입력 → 여러 은닉층(뉴런 묶음)을 거치며 특징을 조합 → 최종 점수 1개를 출력하는 모델
>
> ⚠️ 신경망은 입력 스케일에 민감하므로 **표준화된 데이터**(`_scaled`)를 사용하는 것이 중요함


In [ ]:
# [0] 딥러닝 도구(TensorFlow/Keras) 불러오기 (Colab에는 이미 설치돼 있습니다)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 실행할 때마다 같은 결과가 나오도록 랜덤 시드를 고정합니다
tf.random.set_seed(42)

# [1] 비교 기준: 랜덤 포레스트 회귀 성능을 다시 계산해 둡니다
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_r2 = r2_score(y_test, rf.predict(X_test_scaled))

# [2] DNN(심층 신경망) 모델 만들기
#     - 입력층 : 특성 개수(2054개)만큼 값을 받습니다
#     - 은닉층 : 256 -> 128 -> 64 개의 뉴런 (ReLU 활성화 함수)
#     - Dropout: 일부 뉴런을 잠시 꺼서 과적합(암기)을 막아줍니다
#     - 출력층 : 점수 1개를 예측 (숫자 예측이라 활성화 함수 없음)
model_dnn = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

# [3] 학습 방법 설정 (오차=MSE, 최적화=Adam)
model_dnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)

# [4] 학습시키기
#     - epochs=200      : 데이터를 최대 200번 반복 학습
#     - validation_split: 학습 데이터의 10%를 검증용으로 떼어 성능을 점검
#     - EarlyStopping   : 검증 성능이 20번 이상 나아지지 않으면 조기 종료(과적합 방지)
early_stop = keras.callbacks.EarlyStopping(
    patience=20, restore_best_weights=True
)
history = model_dnn.fit(
    X_train_scaled.values, y_train.values,
    validation_split=0.1,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)
print('DNN 학습 완료! (실제 학습 에폭: {})'.format(len(history.history['loss'])))

# [5] 평가용 데이터로 점수 예측하기
y_pred_dnn = model_dnn.predict(X_test_scaled.values).flatten()

# [6] 성능 지표 계산
r2_dnn = r2_score(y_test, y_pred_dnn)
rmse_dnn = mean_squared_error(y_test, y_pred_dnn) ** 0.5

# [7] 랜덤 포레스트 vs DNN 성능 비교
print('\n=== 성능 비교 (R2는 1에 가까울수록, RMSE는 0에 가까울수록 좋음) ===')
print('Random Forest  R2: {:.3f}'.format(rf_r2))
print('DNN            R2: {:.3f}'.format(r2_dnn))
print('DNN            RMSE: {:.3f}'.format(rmse_dnn))

In [ ]:
# === DNN 성능 시각화 ===
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# [1] 학습 곡선: 에폭이 지날수록 오차(loss)가 줄어드는 모습
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('DNN Training Curve')
axes[0].legend()

# [2] Parity plot: 실제값 vs DNN 예측값 (대각선에 가까울수록 정확)
axes[1].scatter(y_test, y_pred_dnn, alpha=0.5, color='C4')
lo = min(y_test.min(), y_pred_dnn.min())
hi = max(y_test.max(), y_pred_dnn.max())
axes[1].plot([lo, hi], [lo, hi], color='gray', lw=1, linestyle='--', label='y = x')
axes[1].set_xlabel('Actual Score')
axes[1].set_ylabel('Predicted Score')
axes[1].set_title('DNN Parity Plot (R2 = {:.3f})'.format(r2_dnn))
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.show()